<p align="center">
<a href="https://duckietown.com"><img src="../assets/images/dtlogo.png" alt="Duckietown Logo" width="50%"></a>
</p>


<style>
.lx-table {
  margin: 1.5em auto;
  text-align: left;
}

.lx-table caption {
  caption-side: top;
  font-size: 0.9em;
  margin-bottom: 0.6em;
  text-align: center;
}

.lx-figure {
  margin: 1.5em auto;
  text-align: center;
}

.lx-figure figcaption {
  font-size: 0.9em;
  margin-top: 0.6em;
  text-align: center;
}

table {
  margin: 0 auto 1.5em;
}

p:has(> a[id^='table-']) {
  margin: 0;
}

p:has(> a[id^='table-']) + p,
a[id^='table-'] + p {
  font-size: 0.9em;
  margin: 1.5em 0 0.6em;
  text-align: center;
}

p.lx-figure {
  margin: 1.5em auto 0;
}

p.lx-figure + p {
  font-size: 0.9em;
  margin: 0.6em 0 1.5em;
  text-align: center;
}
</style>

# Duckiedrone ToF Architecture

A Duckiedrone does not deliver a downward range reading straight from the sensor chip to a ROS 2 subscriber. This notebook traces the programs that carry and modify it along the way.

The ToF-specific path carries a range reading from the sensor over [Inter-Integrated Circuit (I2C)](https://en.wikipedia.org/wiki/I%C2%B2C) to its driver, then through [Duckietown Postal Service (DTPS)](https://docs.duckietown.com/ente/duckietown-manual/04-software-tools/duckietown-postal-service-dtps.html) and a ROS 2 bridge. The lower flight-control branch sends the same reading through [MAVROS2](https://github.com/mavlink/mavros) over [Micro Air Vehicle Link (MAVLink)](https://mavlink.io/) to [PX4](https://docs.px4.io/main/)'s [Extended Kalman Filter 2 (EKF2)](https://docs.px4.io/main/en/advanced_config/tuning_the_ecl_ekf.html). Each stage can change the reading. The same programs run on a virtual Duckiedrone, where the readings are simulated instead of measured.

By the end of the notebook you should be able to answer:

- Which pieces of software handle a range reading between the chip and a subscriber?

- What does each one add to, or remove from, the reading?

- What sets how often a new reading arrives?

- Why do a target the sensor cannot detect and a target beyond $3.6\,\mathrm{m}$ arrive on the topic as the
  same value?

- Why does stopping the learner subscriber not stop the flight-control path, while a ToF driver or
  bridge failure stops both?

## The big picture

This notebook follows the downward sensor, which runs whenever the Duckiedrone is on. [Figure 1](#figure-1) shows its configured $30\,\mathrm{Hz}$ path to the subscriber. Actual timing can differ, particularly on a virtual Duckiedrone. [Table 3](#table-3) lists the configured rates and startup state of all five sensors.

<figure id="figure-1" class="lx-figure">
  <pre style="display:inline-block; margin:0; max-width:100%; overflow-x:auto; text-align:left;">
              I2C                        DTPS                      ROS 2
+----------+ 30 Hz +-------------------+ 30 Hz +-----------------+ 30 Hz +-----------------+
| VL53L1X  |------>| ToF driver        |------>| ROS 2 bridge    |------>| Subscriber node |
| (bottom) |       | driver-tof-bottom |       | ros2-tof-bottom |       +-----------------+
+----------+       +-------------------+       +-----------------+
                                                        |
                                                        | ROS 2, 30 Hz
                                                        v
                                           +------------------------+
                                           | MAVROS2                |
                                           | ros2-mavros            |
                                           | distance_sensor plugin |
                                           +------------------------+
                                                        |
                                                        | MAVLink DISTANCE_SENSOR, 30 Hz
                                                        v
                                              +-------------------+
                                              | Flight controller |
                                              | PX4, EKF2         |
                                              +-------------------+</pre>
  <figcaption>
    Figure 1: Downward ToF data flow from the VL53L1X to a subscriber node and the flight
    controller.
  </figcaption>
</figure>

The `Subscriber node` box represents the program built in [Notebook 3](./3-subscribing-to-tof.ipynb). The branch below the bridge supplies the separate flight-control path; this notebook follows the subscriber branch.

On a physical Duckiedrone, the driver and bridge run on the Raspberry Pi. On a virtual Duckiedrone, they run on the hosting computer and receive simulated readings instead of measurements from a chip.

## From the VL53L1X to the Raspberry Pi

Each VL53L1X connects to the Raspberry Pi over I2C, a two-wire bus on which the computer addresses each device by a number. The chip does all of the work described in [Notebook 1](./1-tof-basics.ipynb) on its own: it fires the pulses, builds the histogram, finds the peak, and holds the result, in $\mathrm{mm}$, until the Raspberry Pi reads it.

Every VL53L1X starts up at the same I2C address, `0x29`. The address can be changed, but only while that sensor is the one answering at `0x29`, and the change is lost when power is removed, so sharing one bus would mean bringing the five sensors up one at a time on every boot. The Duckiedrone's sensor board avoids that with an I2C multiplexer, a [TCA9548A](https://www.ti.com/product/TCA9548A), which gives each sensor a channel of its own. Linux presents each channel to the Raspberry Pi as a separate bus.

[Table 1](#table-1) lists the I2C channel assigned to each ToF sensor.

<a id="table-1"></a>

Table 1: I2C channels assigned to the Duckiedrone ToF sensors.

| Sensor | Port | I2C bus | Points |
| --- | --- | --- | --- |
| `bottom` | `CHL0` | $23$ | $+z$ (down) |
| `left` | `CHL1` | $24$ | $-y$ (left) |
| `right` | `CHL2` | $25$ | $+y$ (right) |
| `top` | `CHL3` | $26$ | $-z$ (up) |
| `front` | `CHL4` | $27$ | $+x$ (forward) |

The directions use the Duckiedrone body axes: $+x$ is forward, $+y$ right, and $+z$ down. The camera faces $+x$.

## The driver

One container per sensor runs the program that reads the chip, named `driver-tof-bottom`, `driver-tof-front`, and so on. At startup it opens the sensor's bus and programs the settings from [Notebook 1](./1-tof-basics.ipynb): the distance mode, the timing budget, and the inter-measurement period. It also announces the sensor's fixed properties once: its field of view, and the minimum and maximum distance it is specified for in that mode.

Then it repeats the same steps for every measurement:

1. __It waits for a new measurement.__ The chip signals when a measurement is ready. Without
   that check, the driver would read the previous result again and publish it as if it were new.

2. __It converts to $\mathrm{m}$.__ The chip reports $\mathrm{mm}$, while ROS and the rest of the
   Duckietown software work in $\mathrm{m}$.

3. __It drops unusable readings.__ Two cases leave the driver with no distance to report: the
   chip found no peak that cleared the noise, or the distance is beyond the mode's maximum,
   $3.6\,\mathrm{m}$ in `LONG_RANGE`. In both cases the driver publishes a reading with no value.

4. __It stamps the reading__ with the time it read the measurement.

5. __It publishes on DTPS__ at `sensor/time_of_flight/<sensor_name>/range`.

On a virtual Duckiedrone this step is where the difference lies. There is no chip and no I2C bus, so the Duckiematrix connects to the same DTPS path and supplies readings computed by the simulator. The Duckiematrix has to be running, with the Duckiedrone attached to an entity, that is, linked to its counterpart in the simulated world; otherwise the driver is running but has nothing to publish.

## The ROS 2 bridge

A second container per sensor, `ros2-tof-bottom` for the downward one, receives each DTPS message and republishes it as a standard ROS 2 message of type `sensor_msgs/Range`, on the topic:

```text
/DUCKIEDRONE_NAME/bottom_tof_driver_node/range
```

The `range` field has to hold a number, so the bridge fills a reading with no value with `999.0`. It does the same for any distance above `max_range`. The value is not arbitrary. The [message definition](https://github.com/ros2/common_interfaces/blob/jazzy/sensor_msgs/msg/Range.msg) says of `range`: "values < range_min or > range_max should be discarded" (the fields themselves are named `min_range` and `max_range`), so a number far outside those limits is the message's own way of marking a reading to discard.

This is also why the two cases from [the driver section](#the-driver) look the same on the topic: a target the chip cannot detect and a target beyond $3.6\,\mathrm{m}$ both arrive as `999.0`. Applying the limits is left to whatever reads the topic. The node written in [Notebook 3](./3-subscribing-to-tof.ipynb) prints raw values, so that its output shows readings on both sides of them.

### What the bridge fills in

A `Range` message holds more than a distance. The bridge takes the sensor's fixed properties from the driver's announcement and repeats them in every message.

[Table 2](#table-2) shows the fields published in a `sensor_msgs/Range` message.

<a id="table-2"></a>

Table 2: Fields in the ROS 2 `sensor_msgs/Range` message.

| Field | What arrives | Note |
| --- | --- | --- |
| `header.stamp` | When the driver read the measurement | Carried through from the driver |
| `header.frame_id` | `DUCKIEDRONE_NAME/tof/` | Set by the driver |
| `range` | Reported range in $\mathrm{m}$, or `999.0` | $\tilde{d}$ when the driver has a reading; `999.0` marks no usable range |
| `min_range` | $0.04\,\mathrm{m}$ | Closest specified usable range, not a lower limit on what the chip can report |
| `max_range` | $3.6\,\mathrm{m}$ | The driver's maximum in `LONG_RANGE` |
| `field_of_view` | $0.471\,\mathrm{rad}$ | The $27^\circ$ cone, in $\mathrm{rad}$ |
| `radiation_type` | `1` | The message definition's code for `INFRARED` |
| `variance` | `0.0` | The message reads `0` as "variance unknown", not as "no noise" |

The chip does not timestamp its readings, so the driver stamps each one when it reads it. The bridge keeps that stamp instead of using the time the message reached it, so `header.stamp` is when the driver read the distance, as the message definition asks.

[Notebook 3](./3-subscribing-to-tof.ipynb) prints one of these messages from a Duckiedrone, to check against this table.

## The other consumer: the flight controller

The branch below the bridge in [Figure 1](#figure-1) carries the downward `Range` readings to the flight controller as a height cue. The same bridge also supplies the subscriber written in [Notebook 3](./3-subscribing-to-tof.ipynb).

The downward ToF sensor measures distance along its own axis to the surface below, not altitude above a fixed reference. When the Duckiedrone is level over a bare floor, that distance equals its height above the floor and is useful as a height cue. A tilted Duckiedrone measures a longer, slanted distance, and flying over an object changes the reading even when altitude is unchanged.

If the driver or bridge stops, neither the flight-controller branch nor the subscriber receives new range readings.

## From the bridge to a subscriber

The last stop is a node that subscribes to `/DUCKIEDRONE_NAME/bottom_tof_driver_node/range`. It receives the `Range` message described in [the bridge section](#the-ros-2-bridge), with the conversion to $\mathrm{m}$, the `999.0` substitution, and the timestamp already applied. What it does with each message is up to that node, and [Notebook 3](./3-subscribing-to-tof.ipynb) writes such a node.

## Not all five sensors are treated the same

[Table 3](#table-3) compares the configured rates and startup states of the five ToF sensors.

<a id="table-3"></a>

Table 3: Configured ToF sensor rates and startup state.

| Sensor | Configured rate ($\mathrm{Hz}$) | Running by default |
| --- | --- | --- |
| `bottom` | $30$ | Yes |
| `front` | $5$ | No |
| `left` | $5$ | No |
| `right` | $5$ | No |
| `top` | $5$ | No |

The five sensors differ in two ways. The downward sensor is configured for $30\,\mathrm{Hz}$ and starts with the Duckiedrone, while the other four are configured for $5\,\mathrm{Hz}$ and stay off until someone starts them. Both differences save processor time on the Raspberry Pi, which the camera and the flight software also need. Each sensor runs two programs, a driver and a bridge, so starting all five means ten containers instead of two.

## Changing any of this

Each driver reads its settings from a file on the Duckiedrone:

```text
/data/config/node/tof_driver_<sensor_name>/DUCKIEDRONE_NAME.yaml
```

If the file does not exist when the driver starts, the driver creates it from its defaults. From then on the driver reads only this file, so editing it changes the settings, and a later update to the defaults does not overwrite those edits. The downward sensor's default file, with some comments and entries removed:

```yaml
sensor_name: "bottom"
sensor_model: "VL53L1X"

# publish rate, Hz
frequency: 30

# SHORT_RANGE: less reach, better in bright light. LONG_RANGE: more reach.
mode: "LONG_RANGE"

# photon collection time per measurement, ms: 15 (SHORT_RANGE only), 20, 33,
# 50, 100, 200, or 500. Longer is less noisy and reaches further, but slower.
timing_budget: 33

# time between measurements, ms, at least timing_budget. Set from frequency if omitted.
# inter_measurement_period: 33

display_fragment_frequency: 0
connectors:
  # Raspberry Pi 4, port CHL0
  -   bus: 23
      address: 0x29
```

`mode`, `timing_budget`, and `inter_measurement_period` are the chip settings from [Notebook 1](./1-tof-basics.ipynb). `frequency` belongs to the driver, but it also sets the chip's pace: when `inter_measurement_period` is left out, the driver sets it to the reciprocal of `frequency`, in $\mathrm{s}$, or to the timing budget if that is longer.

The chip cannot finish measurements faster than one per timing budget, or one per inter-measurement period if that is longer. If `frequency` asks for more than the chip can deliver, the driver logs a warning and publishes at the highest rate the chip allows. It does not repeat a measurement to make up the difference, because of the check in [the driver section](#the-driver).

Two mistakes stop the driver at startup with an error: a timing budget the chip does not accept, and an `inter_measurement_period` shorter than the timing budget.

The other four sensors use `frequency: 5` with `timing_budget: 200`, the longest budget that fits in $200\,\mathrm{ms}$. To run one of them faster, shorten its budget too, for example `frequency: 10` with `timing_budget: 100`.

On a physical Duckiedrone the file belongs to root, so edit it with `sudo`. Both the edit and the restart happen on an authorized Duckiedrone. After editing, restart the sensor's two containers, the driver first and then the bridge:

```bash
docker restart driver-tof-bottom
docker restart ros2-tof-bottom
```

The order matters. The bridge subscribes to the driver's DTPS channel once, when it starts, and never subscribes again, so a bridge left running across a driver restart stops publishing.

## Further reading

The [VL53L1X product page](https://www.st.com/en/imaging-and-photonics-solutions/vl53l1x.html) describes the sensor, while the [sensor_msgs/Range message definition](https://github.com/ros2/common_interfaces/blob/jazzy/sensor_msgs/msg/Range.msg) specifies the bridge output.

## Checkpoint

Run the self-check in the next cell. Write or select a response before revealing the answer.


In [ ]:
import sys
from pathlib import Path

working_directory = Path.cwd()
parent_directory = working_directory.parent
if (parent_directory / "packages").is_dir():
    parent_directory_path = str(parent_directory)
    sys.path.insert(0, parent_directory_path)

from packages.checkpoint_self_check import display_checkpoint_self_checks

display_checkpoint_self_checks()